# 5.3. 延后初始化

框架允许在定义网络时不确定输入维度，直到数据第一次通过模型时才推断各层大小并初始化参数。请自己完成每个代码单元。

### 5.3.1. 实例化网络

定义一个多层感知机 get_net()：
- def get_net(): return nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))  （注意 LazyLinear 表示输入维度在第一次前向时按输入推断；若用普通 Linear 则需显式给出 in_features）
- 实例化 net = get_net()，先打印 net 观察（此时权重尚未确定）

然后生成 X = torch.rand(size=(2, 20))，通过 net(X) 触发参数初始化，再打印 net 观察维度是否被推断出来，并查看 net[0].weight.shape。

In [3]:
import torch 
from torch import nn

def get_net():
    return nn.Sequential(nn.LazyLinear(256),nn.ReLU(),nn.LazyLinear(10))
net=get_net()
print(net)
x=torch.rand(2,20)
net(x)
print(net)


Sequential(
  (0): LazyLinear(in_features=0, out_features=256, bias=True)
  (1): ReLU()
  (2): LazyLinear(in_features=0, out_features=10, bias=True)
)
Sequential(
  (0): Linear(in_features=20, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=10, bias=True)
)


（补充理解）在 PyTorch 中，若使用 nn.Linear，则输入维度必须在构造时指定；延后初始化主要由 LazyLinear / LazyModule 提供。试对比 net = nn.Sequential(nn.Linear(?, 256), nn.ReLU(), nn.LazyLinear(10)) 对第一层传入 X 后各层参数形状的变化。

In [2]:
net_2=nn.Sequential(nn.Linear(10, 256), nn.ReLU(), nn.LazyLinear(10))
print(net_2)

Sequential(
  (0): Linear(in_features=10, out_features=256, bias=True)
  (1): ReLU()
  (2): LazyLinear(in_features=0, out_features=10, bias=True)
)


### 5.3.2. 小结

- 延后初始化使框架能自动推断参数形状，简化模型定义和修改
- 首次将数据通过模型后，框架会按顺序初始化各层参数

### 5.3.3. 练习

1. 如果指定了第一层的输入尺寸，但未指定后续层的尺寸，会发生什么？是否立即初始化？
2. 如果指定了不匹配的维度会发生什么？
3. 如果输入具有不同的维度，需要做什么？（提示：考察参数绑定）